# Bajaj Capital — Knowledge Base Pipeline (.txt, run cell-by-cell)

Builds a searchable vector index from **.txt** files organized in section folders,
then queries it. Backs the `search_knowledge_base` tool in your GPT-OSS fine-tune
(replaces the FAKE stub in cell 16 of the training notebook).

**Expected folder layout** (folder name == section name):

```
knowledge_base/
    LI_Policy/      <- ~50 .txt files
    HI_Policy/      <- ~50 .txt files
    LI_Brochure/    <- ~50 .txt files
```

Run cells top to bottom. Cells 1-6 only need to run **once** to build the index;
after that, jump straight to the query / tool cells (set RESET = False).

### 1. Install dependencies

In [14]:
!pip3 install -q sentence-transformers chromadb

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


You should consider upgrading via the '/Users/admin/Desktop/mini-backend/.venv/bin/python3 -m pip install --upgrade pip' command.


In [15]:
!pip3 install "numpy<2" "transformers<4.44" "sentence-transformers<3.1" "tokenizers<0.20"

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


You should consider upgrading via the '/Users/admin/Desktop/mini-backend/.venv/bin/python3 -m pip install --upgrade pip' command.


### 2. Config
Edit these, then run.

In [16]:
import os, re, sys, json
from pathlib import Path

# --- paths ---
ROOT_DIR = "/Users/admin/Desktop/mini-backend/knowledge_v2/generated_kb"   # folder that CONTAINS the section subfolders
DB_PATH  = "./kb_chroma"        # where the persistent index is stored
RESET    = True                 # True = drop & rebuild the index from scratch

# --- model ---
# bge-base-en-v1.5: strong, small retrieval model (~400MB on first download).
# Swap to "BAAI/bge-small-en-v1.5" for ~2x speed, or "BAAI/bge-m3" for multilingual.
EMBED_MODEL = "BAAI/bge-base-en-v1.5"
QUERY_INSTRUCTION = "Represent this sentence for searching relevant passages: "

# --- device & batching ---
# "cpu" is the safe default and finishes a one-time build in a few minutes.
# Switch to "mps" (Apple GPU) for speed ONLY if you have free memory; keep the
# batch small (8) if you do, or you may hit "MPS backend out of memory".
DEVICE = "cpu"            # "cpu" | "mps" | "cuda"
EMBED_BATCH = 8           # peak GPU/CPU memory scales with this; lower if OOM

# --- chunking (bge has a hard 512-token limit; keep chunks safely under it) ---
MAX_TOKENS, OVERLAP_TOKENS = 450, 60
ADD_BATCH = 256
COLLECTION_NAME = "bajaj_kb"

print("config set. root =", ROOT_DIR, "| db =", DB_PATH)

config set. root = /Users/admin/Desktop/mini-backend/knowledge_v2/generated_kb | db = ./kb_chroma


### 3. Load the embedding model
First run downloads the model — give it a minute.

In [17]:
import transformers
transformers.logging.set_verbosity_error()  # silence the harmless 1167>512 notice
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(EMBED_MODEL, device=DEVICE)
model.max_seq_length = MAX_TOKENS            # hard truncation backstop
_tok = model.tokenizer
count_tokens = lambda s: len(_tok.encode(s, add_special_tokens=False))
print("loaded:", EMBED_MODEL, "| device:", model.device)

loaded: BAAI/bge-base-en-v1.5 | device: cpu


In [18]:
!pip3 install chromadb

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


You should consider upgrading via the '/Users/admin/Desktop/mini-backend/.venv/bin/python3 -m pip install --upgrade pip' command.


### 4. Open / create the vector index

In [19]:
import chromadb

client = chromadb.PersistentClient(path=DB_PATH)
if RESET:
    try:
        client.delete_collection(COLLECTION_NAME)
        print("reset: dropped existing collection")
    except Exception:
        pass
col = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},  # pairs with normalized embeddings
)
print("collection ready:", COLLECTION_NAME, "| current count:", col.count())

reset: dropped existing collection
collection ready: bajaj_kb | current count: 0


### 5. Read + chunk helpers

In [20]:
def read_text_file(path):
    """Read a .txt file as UTF-8, replacing any undecodable bytes."""
    return Path(path).read_text(encoding="utf-8", errors="replace")

import json

def read_json_file(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    return json.dumps(
        data,
        ensure_ascii=False,
        indent=2
    )

def _atomic_units(text, max_tokens):
    """Break text into units each under max_tokens
    (paragraphs -> sentences -> words as last resort)."""
    blocks = [b.strip() for b in re.split(r"\n\s*\n", text) if b.strip()]
    units = []
    for b in blocks:
        b = re.sub(r"[ \t]+", " ", b)
        if count_tokens(b) <= max_tokens:
            units.append(b); continue
        for s in re.split(r"(?<=[.!?])\s+", b):
            s = s.strip()
            if not s:
                continue
            if count_tokens(s) <= max_tokens:
                units.append(s)
            else:  # very long single sentence -> hard word split
                cur = []
                for w in s.split():
                    cur.append(w)
                    if count_tokens(" ".join(cur)) >= max_tokens:
                        units.append(" ".join(cur)); cur = []
                if cur:
                    units.append(" ".join(cur))
    return units

def chunk_text(text, max_tokens=MAX_TOKENS, overlap_tokens=OVERLAP_TOKENS):
    """Greedily pack units into ~max_tokens chunks with a small token overlap."""
    units = _atomic_units(text, max_tokens)
    chunks, cur, cur_tok = [], [], 0
    for u in units:
        utok = count_tokens(u)
        if cur and cur_tok + utok > max_tokens:
            chunks.append(" ".join(cur))
            tail, tail_tok = [], 0
            for x in reversed(cur):
                xt = count_tokens(x)
                if tail_tok + xt > overlap_tokens:
                    break
                tail.insert(0, x); tail_tok += xt
            cur, cur_tok = list(tail), tail_tok
        cur.append(u); cur_tok += utok
    if cur:
        chunks.append(" ".join(cur))
    return chunks

print("helpers defined")

helpers defined


### 6. Build the index
Walks each section folder, reads every .txt, chunks, embeds, stores. Run once.

In [21]:
root = Path(ROOT_DIR)
assert root.is_dir(), f"root folder not found: {root}"
sections = sorted(p for p in root.iterdir() if p.is_dir())
assert sections, f"no section subfolders inside {root}"
print("sections:", [s.name for s in sections])

ids, docs, metas = [], [], []
totals = {"files": 0, "chunks": 0}

def flush():
    if not ids:
        return
    embs = model.encode(docs, batch_size=EMBED_BATCH,
                        normalize_embeddings=True, show_progress_bar=False).tolist()
    col.add(ids=ids, embeddings=embs, documents=docs, metadatas=metas)
    ids.clear(); docs.clear(); metas.clear()
    if DEVICE == "mps":
        import torch
        if hasattr(torch, "mps"):
            torch.mps.empty_cache()

for section_dir in sections:
    print(section_dir)
    print(list(section_dir.iterdir())[:5])
    section = section_dir.name           # folder name IS the section label
    files = list(section_dir.glob("*.txt"))
    files += list(section_dir.glob("*.json"))
    files = sorted(files)
    print(f"  {section}: {len(files)} txt file(s)")
    for fp in files:
        totals["files"] += 1
        if fp.suffix.lower() == ".txt":
            text = read_text_file(fp)

        elif fp.suffix.lower() == ".json":
            text = read_json_file(fp)

        else:
            continue

        for ci, chunk in enumerate(chunk_text(text)):
            ids.append(f"{section}::{fp.name}::c{ci}")
            docs.append(chunk)
            metas.append({"section": section, "source_file": fp.name, "chunk_index": ci})
            totals["chunks"] += 1
            if len(ids) >= ADD_BATCH:
                flush()
flush()
print(f"\nDONE -> {totals['chunks']} chunks across {totals['files']} files. "
      f"count in index: {col.count()}")

sections: ['Hi Document', 'Hi__Document', 'LI Plan Brochure', 'LI Policy Bond Documents', 'LI__Plan__Brochure', 'LI__Policy__Bond__Documents', 'manual_kb']
/Users/admin/Desktop/mini-backend/knowledge_v2/generated_kb/Hi Document
[PosixPath('/Users/admin/Desktop/mini-backend/knowledge_v2/generated_kb/Hi Document/secure---policy-terms-and-conditions-(effective-from-05-march-2025).json'), PosixPath('/Users/admin/Desktop/mini-backend/knowledge_v2/generated_kb/Hi Document/care-supreme---policy-terms-&-conditions-(effective-from-19-march-2025) (1).json'), PosixPath('/Users/admin/Desktop/mini-backend/knowledge_v2/generated_kb/Hi Document/Care Advantage final ver April 2025.json'), PosixPath('/Users/admin/Desktop/mini-backend/knowledge_v2/generated_kb/Hi Document/3ae97e2b9fc6f41ecdb07c38820229b6269faaa1.json'), PosixPath('/Users/admin/Desktop/mini-backend/knowledge_v2/generated_kb/Hi Document/Care Supreme-Senior Citizen options.json')]
  Hi Document: 46 txt file(s)
/Users/admin/Desktop/mini-bac

### 7. The retrieval function (this is your tool body)

In [22]:
def search_knowledge_base(query: str, section: str = "all", k: int = 4) -> str:
    """Search the Bajaj Capital knowledge base for product or process information.

    Args:
        query: The customer's question or search terms.
        section: One of "LI_Policy", "HI_Policy", "LI_Brochure", or "all".
    """
    q_emb = model.encode([QUERY_INSTRUCTION + query],
                         normalize_embeddings=True)[0].tolist()
    where = None if (not section or section == "all") else {"section": section}
    res = col.query(query_embeddings=[q_emb], n_results=k, where=where)

    docs_ = res.get("documents", [[]])[0]
    metas_ = res.get("metadatas", [[]])[0]
    dists_ = res.get("distances", [[]])[0]
    if not docs_:
        return json.dumps({"found": False, "content": "No relevant KB entry found."},
                          ensure_ascii=False)

    results = [{
        "content": d,
        "source_file": m.get("source_file"),
        "chunk_index": m.get("chunk_index"),
        "section": m.get("section"),
        "score": round(1 - dist, 4),
    } for d, m, dist in zip(docs_, metas_, dists_)]
    return json.dumps({"found": True, "section": section, "results": results},
                      ensure_ascii=False)

print("search_knowledge_base() ready")

search_knowledge_base() ready


### 8. Test a query
Tune `section` / `k` and eyeball the chunks before wiring into training.

In [23]:
out = search_knowledge_base(
    "what is term insurance?",
    section="all",   # or "HI_Policy", "LI_Brochure", "all"
    k=4,
)
print(json.dumps(json.loads(out), indent=2, ensure_ascii=False))

{
  "found": true,
  "section": "all",
  "results": [
    {
      "content": "\"question\": \"What is policy term?\", \"answer\": \"Policy term is the duration for which the insurance policy remains in force.\" }, { \"question\": \"What is surrender value?\", \"answer\": \"Surrender value is the amount payable if a policyholder voluntarily terminates an eligible policy before maturity.\" }, { \"question\": \"What is grace period?\", \"answer\": \"Grace period is the extra time allowed after the premium due date during which the policy remains active and the premium can still be paid.\" }, { \"question\": \"What is free look period?\", \"answer\": \"Free look period is the period during which a new policyholder can review and cancel the policy if dissatisfied, subject to insurer rules.\" }, { \"question\": \"What is term insurance?\", \"answer\": \"Term insurance is a life insurance policy that provides financial protection for a specified period. If the insured person dies during the p

In [24]:
print("count in collection:", col.count())
# what section labels actually exist?
print("sample metadata:", col.get(limit=5, include=["metadatas"])["metadatas"])
# query with NO section filter:
print(search_knowledge_base("wellness discount on premium by walking", section="all", k=4))

count in collection: 7286
sample metadata: [{'section': 'Hi Document', 'source_file': '207161c1b524dda2706f7c6f436557c7291b6f19.json', 'chunk_index': 0}, {'source_file': '207161c1b524dda2706f7c6f436557c7291b6f19.json', 'chunk_index': 1, 'section': 'Hi Document'}, {'source_file': '207161c1b524dda2706f7c6f436557c7291b6f19.json', 'section': 'Hi Document', 'chunk_index': 2}, {'source_file': '207161c1b524dda2706f7c6f436557c7291b6f19.json', 'chunk_index': 3, 'section': 'Hi Document'}, {'chunk_index': 4, 'source_file': '207161c1b524dda2706f7c6f436557c7291b6f19.json', 'section': 'Hi Document'}]
{"found": true, "section": "all", "results": [{"content": "Residual points will be carried forward to the next Policy \nYear and accrued with that Policy Year’s Wellness Points. Hence, these points are not lost. iii. Discount is on the individual’s premium in Individual plan \nand on Floater Policy Premium in Floater plans. Discount \nwill be considered only for Insured Persons aged 18 years \nand above

### 9. Wiring into the training notebook

In **cell 16** of `train_runpod_with_tooling`, replace the FAKE stub body with the
`search_knowledge_base` defined here (same signature, real retrieval). Make sure the
tool's `section` enum in its docstring/schema reads:

```
"LI_Policy" / "HI_Policy" / "LI_Brochure" / "all"
```

so the values the model trains on exactly match what it must pass at runtime.

Since your sources are already plain text, there is no extraction step to worry
about. The one thing worth checking is **encoding**: if any file shows replacement
characters (`\ufffd`) in the test output, it wasn't UTF-8 — re-save it as UTF-8 and
rebuild. If a single .txt actually contains several distinct documents concatenated,
consider splitting it so retrieval can cite the right source.

In [25]:
import os
for r, _, fs in os.walk("kb_chroma"):
    for f in fs:
        p = os.path.join(r, f)
        print(f"{os.path.getsize(p):>12,}  {p}")

 104,202,240  kb_chroma/chroma.sqlite3
  23,023,616  kb_chroma/3d719d82-87a4-4750-ab7c-912977ab6df7/data_level0.bin
      28,672  kb_chroma/3d719d82-87a4-4750-ab7c-912977ab6df7/length.bin
      62,808  kb_chroma/3d719d82-87a4-4750-ab7c-912977ab6df7/link_lists.bin
         100  kb_chroma/3d719d82-87a4-4750-ab7c-912977ab6df7/header.bin
   1,216,380  kb_chroma/3d719d82-87a4-4750-ab7c-912977ab6df7/index_metadata.pickle
   4,461,468  kb_chroma/83146684-1d89-4b0a-b848-be197c38ef91/data_level0.bin
       5,556  kb_chroma/83146684-1d89-4b0a-b848-be197c38ef91/length.bin
      12,356  kb_chroma/83146684-1d89-4b0a-b848-be197c38ef91/link_lists.bin
         100  kb_chroma/83146684-1d89-4b0a-b848-be197c38ef91/header.bin
     224,128  kb_chroma/83146684-1d89-4b0a-b848-be197c38ef91/index_metadata.pickle


In [26]:
import json

questions = [
    "What is term insurance?",
    "What is waiting period?",
    "What is deductible?",
    "What is cashless claim?",
    "What is sum insured?",
    "What is Care Supreme?",
    "What is waiting period for pre existing disease?",
    "What is cumulative bonus?",
    "What is room rent limit?",
    "What is maternity cover?",
    "What is deductible option?",
    "What is claim settlement process?",
    "What is term insurance?",
    "What is life insurance?",
    "What is ULIP?",
    "What is nominee?",
    "What is maturity benefit?",
    "What is Bajaj Capital?",
    "What services does Bajaj Capital provide?",
    "How do I file a claim?",
    "What is deductible?"
]

for q in questions:

    print("\n")
    print("="*100)
    print("QUESTION:", q)

    res = json.loads(
        search_knowledge_base(q, section="all", k=5)
    )

    for i, chunk in enumerate(res["results"]):

        print("\nRESULT", i+1)
        print("Score:", chunk["score"])
        print("File :", chunk["source_file"])
        print(chunk["content"][:500])



QUESTION: What is term insurance?

RESULT 1
Score: 0.6641
File : insurance_glossary.json
"question": "What is policy term?", "answer": "Policy term is the duration for which the insurance policy remains in force." }, { "question": "What is surrender value?", "answer": "Surrender value is the amount payable if a policyholder voluntarily terminates an eligible policy before maturity." }, { "question": "What is grace period?", "answer": "Grace period is the extra time allowed after the premium due date during which the policy remains active and the premium can still be paid." }, { "que

RESULT 2
Score: 0.6503
File : insurance_glossary.json
Part of the premium provides insurance cover, while the remaining amount is invested in funds chosen by the policyholder."
 },
 {
 "question": "What is an endowment plan?",
 "answer": "An endowment plan is a life insurance policy that provides both life cover and savings. It pays a maturity benefit if the policyholder survives the policy term and a de